In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

from utils import (
    get_hdfs_base_uri,
    create_namespace,
    clean_string,
    clean_upper,
    only_digits,
    parse_long,
    parse_date,
    add_record_hash,
    add_silver_metadata,
    validate_expected_columns,
    get_latest_batch_id,
    build_null_condition,
    build_rejection_reason,
    deduplicate_by_keys,
    merge_iceberg_by_keys,
    replace_iceberg_by_partition_values,
    VALID_UFS,
    default_spark_local_dir,
    is_truthy,
)

SPARK_LOCAL_DIR = default_spark_local_dir()
os.environ["SPARK_LOCAL_DIRS"] = SPARK_LOCAL_DIR
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .getOrCreate()
)

# ============================================================
# Escolha arquitetural da Silver
# ============================================================
# Silver = camada limpa, padronizada, source-aligned e ainda granular.
# Aqui sim pode haver deduplicação, mas com regra explícita:
# chave natural/de negócio + critério determinístico de desempate.
#
# A Silver não cria surrogate keys dimensionais com hash.
# _record_hash continua existindo apenas como fingerprint técnico para
# detectar alteração de conteúdo em MERGE SCD1 ou auditoria.
#
# Para dados ANS consolidados por competência, assumimos snapshot completo
# por id_cmpt_movel. Por isso, o movimento pode ser substituído por competência,
# o que trata mudança/desaparecimento de chaves naturais dentro da competência.
# Se sua origem virar CDC/evento incremental, defina FULL_SNAPSHOT_REPLACE=false
# e use MERGE por chave natural.

# ============================================================
# Configurações
# ============================================================

HDFS_BASE_URI = get_hdfs_base_uri()

CATALOG = "spark_catalog"

BRONZE_DB = "bronze"
SILVER_DB = "silver"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_DB}.beneficiarios"

SILVER_OPERADORA_TABLE = f"{CATALOG}.{SILVER_DB}.operadora"
SILVER_MUNICIPIO_TABLE = f"{CATALOG}.{SILVER_DB}.municipio"
SILVER_PLANO_TABLE = f"{CATALOG}.{SILVER_DB}.plano"
SILVER_MOVIMENTO_TABLE = f"{CATALOG}.{SILVER_DB}.beneficiario_movimento"
SILVER_REJEITADOS_TABLE = f"{CATALOG}.{SILVER_DB}.beneficiario_rejeitado"

SILVER_LOCATION = f"{HDFS_BASE_URI}/user/hive/warehouse/{SILVER_DB}.db"

FULL_SNAPSHOT_REPLACE = is_truthy(os.getenv("FULL_SNAPSHOT_REPLACE", "true"))

create_namespace(
    spark=spark,
    catalog=CATALOG,
    database=SILVER_DB,
    location=SILVER_LOCATION,
)

BATCH_ID = os.getenv("BATCH_ID")

if not BATCH_ID:
    BATCH_ID = get_latest_batch_id(
        spark=spark,
        table_name=BRONZE_TABLE,
        batch_column="_batch_id",
    )

print(f"Processando batch: {BATCH_ID}")

# ============================================================
# Leitura incremental da Bronze
# ============================================================

df_bronze = (
    spark.table(BRONZE_TABLE)
    .where(F.col("_batch_id") == BATCH_ID)
)

EXPECTED_COLUMNS = [
    "_record_hash",
    "_batch_id",
    "id_cmpt_movel",
    "cd_operadora",
    "nm_razao_social",
    "nr_cnpj",
    "modalidade_operadora",
    "sg_uf",
    "cd_municipio",
    "nm_municipio",
    "tp_sexo",
    "de_faixa_etaria",
    "de_faixa_etaria_reaj",
    "cd_plano",
    "tp_vigencia_plano",
    "de_contratacao_plano",
    "de_segmentacao_plano",
    "de_abrg_geografica_plano",
    "cobertura_assist_plan",
    "tipo_vinculo",
    "qt_beneficiario_ativo",
    "qt_beneficiario_aderido",
    "qt_beneficiario_cancelado",
    "dt_carga",
    "_source_path",
    "_ingested_at",
    "_source_system",
]

validate_expected_columns(df_bronze, EXPECTED_COLUMNS)

df_bronze = df_bronze.select(*EXPECTED_COLUMNS)

if df_bronze.limit(1).count() == 0:
    raise ValueError(f"Nenhum registro encontrado na Bronze para o batch {BATCH_ID}.")

# ============================================================
# Limpeza e padronização
# ============================================================

df_clean = (
    df_bronze

    # Fingerprint técnico da Bronze, não é chave de negócio.
    .withColumn("_bronze_record_hash", clean_string("_record_hash"))
    .withColumn("_batch_id", clean_string("_batch_id"))

    # Identificadores naturais/de negócio
    .withColumn("id_cmpt_movel", clean_string("id_cmpt_movel"))
    .withColumn("cd_operadora", clean_string("cd_operadora"))
    .withColumn("cd_municipio", clean_string("cd_municipio"))
    .withColumn("cd_plano", clean_string("cd_plano"))

    # Operadora
    .withColumn("nm_razao_social", clean_upper("nm_razao_social"))
    .withColumn("nr_cnpj", only_digits("nr_cnpj"))
    .withColumn("modalidade_operadora", clean_upper("modalidade_operadora"))

    # Município
    .withColumn("sg_uf", clean_upper("sg_uf"))
    .withColumn(
        "sg_uf",
        F.when(F.col("sg_uf").isin(VALID_UFS), F.col("sg_uf"))
        .otherwise(F.lit(None).cast("string"))
    )
    .withColumn("nm_municipio", clean_upper("nm_municipio"))

    # Perfil / plano
    .withColumn("tp_sexo", clean_upper("tp_sexo"))
    .withColumn("de_faixa_etaria", clean_string("de_faixa_etaria"))
    .withColumn("de_faixa_etaria_reaj", clean_string("de_faixa_etaria_reaj"))
    .withColumn("tp_vigencia_plano", clean_upper("tp_vigencia_plano"))
    .withColumn("de_contratacao_plano", clean_upper("de_contratacao_plano"))
    .withColumn("de_segmentacao_plano", clean_upper("de_segmentacao_plano"))
    .withColumn("de_abrg_geografica_plano", clean_upper("de_abrg_geografica_plano"))
    .withColumn("cobertura_assist_plan", clean_upper("cobertura_assist_plan"))
    .withColumn("tipo_vinculo", clean_upper("tipo_vinculo"))

    # Métricas
    .withColumn("qt_beneficiario_ativo", parse_long("qt_beneficiario_ativo"))
    .withColumn("qt_beneficiario_aderido", parse_long("qt_beneficiario_aderido"))
    .withColumn("qt_beneficiario_cancelado", parse_long("qt_beneficiario_cancelado"))

    # Data da carga da fonte
    .withColumn("dt_carga", parse_date("dt_carga"))

    # Metadados
    .withColumn("_source_path", clean_string("_source_path"))
    .withColumn("_bronze_ingested_at", F.col("_ingested_at").cast("timestamp"))
    .withColumn("_source_system", clean_upper("_source_system"))
)

df_clean = add_silver_metadata(
    df=df_clean,
    batch_id=BATCH_ID,
)

# ============================================================
# Qualidade: registros rejeitados
# ============================================================

MOVIMENTO_BUSINESS_KEY = [
    "id_cmpt_movel",
    "cd_operadora",
    "cd_municipio",
    "cd_plano",
    "tp_sexo",
    "de_faixa_etaria",
    "de_faixa_etaria_reaj",
    "tipo_vinculo",
]

REQUIRED_COLUMNS = MOVIMENTO_BUSINESS_KEY

invalid_condition = build_null_condition(REQUIRED_COLUMNS)

df_rejeitados = (
    df_clean
    .filter(invalid_condition)
    .withColumn("_rejection_reason", build_rejection_reason(REQUIRED_COLUMNS))
    .withColumn("_rejected_at", F.current_timestamp())
)

df_valid = df_clean.filter(~invalid_condition)

# ============================================================
# Deduplicação determinística do lote válido
# ============================================================

DEDUPE_ORDER_COLUMNS = [
    "dt_carga",
    "_bronze_ingested_at",
    "_source_path",
    "_bronze_record_hash",
]

df_movimento_base = deduplicate_by_keys(
    df=df_valid,
    key_columns=MOVIMENTO_BUSINESS_KEY,
    order_columns=DEDUPE_ORDER_COLUMNS,
)

df_operadora_base = deduplicate_by_keys(
    df=df_valid.where(F.col("cd_operadora").isNotNull()),
    key_columns=["cd_operadora"],
    order_columns=DEDUPE_ORDER_COLUMNS,
)

df_municipio_base = deduplicate_by_keys(
    df=df_valid.where(F.col("cd_municipio").isNotNull()),
    key_columns=["cd_municipio"],
    order_columns=DEDUPE_ORDER_COLUMNS,
)

df_plano_base = deduplicate_by_keys(
    df=df_valid.where(
        F.col("cd_operadora").isNotNull()
        & F.col("cd_plano").isNotNull()
    ),
    key_columns=["cd_operadora", "cd_plano"],
    order_columns=DEDUPE_ORDER_COLUMNS,
)

# ============================================================
# silver.operadora
# ============================================================

df_operadora = (
    df_operadora_base
    .select(
        "cd_operadora",
        "nm_razao_social",
        "nr_cnpj",
        "modalidade_operadora",
        "_source_system",
        "_batch_id",
        "_silver_ingested_at",
        "_layer",
    )
)

df_operadora = add_record_hash(
    df=df_operadora,
    payload_columns=[
        "cd_operadora",
        "nm_razao_social",
        "nr_cnpj",
        "modalidade_operadora",
        "_source_system",
    ],
    hash_column_name="_record_hash",
)

# ============================================================
# silver.municipio
# ============================================================

df_municipio = (
    df_municipio_base
    .select(
        "cd_municipio",
        "nm_municipio",
        "sg_uf",
        "_source_system",
        "_batch_id",
        "_silver_ingested_at",
        "_layer",
    )
)

df_municipio = add_record_hash(
    df=df_municipio,
    payload_columns=[
        "cd_municipio",
        "nm_municipio",
        "sg_uf",
        "_source_system",
    ],
    hash_column_name="_record_hash",
)

# ============================================================
# silver.plano
# ============================================================

df_plano = (
    df_plano_base
    .select(
        "cd_operadora",
        "cd_plano",
        "tp_vigencia_plano",
        "de_contratacao_plano",
        "de_segmentacao_plano",
        "de_abrg_geografica_plano",
        "cobertura_assist_plan",
        "_source_system",
        "_batch_id",
        "_silver_ingested_at",
        "_layer",
    )
)

df_plano = add_record_hash(
    df=df_plano,
    payload_columns=[
        "cd_operadora",
        "cd_plano",
        "tp_vigencia_plano",
        "de_contratacao_plano",
        "de_segmentacao_plano",
        "de_abrg_geografica_plano",
        "cobertura_assist_plan",
        "_source_system",
    ],
    hash_column_name="_record_hash",
)

# ============================================================
# silver.beneficiario_movimento
# ============================================================

df_movimento = (
    df_movimento_base
    .select(
        "id_cmpt_movel",
        "cd_operadora",
        "cd_municipio",
        "cd_plano",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
        "qt_beneficiario_ativo",
        "qt_beneficiario_aderido",
        "qt_beneficiario_cancelado",
        "dt_carga",
        "_source_path",
        "_bronze_record_hash",
        "_bronze_ingested_at",
        "_source_system",
        "_batch_id",
        "_silver_ingested_at",
        "_layer",
    )
)

df_movimento = add_record_hash(
    df=df_movimento,
    payload_columns=[
        "id_cmpt_movel",
        "cd_operadora",
        "cd_municipio",
        "cd_plano",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
        "qt_beneficiario_ativo",
        "qt_beneficiario_aderido",
        "qt_beneficiario_cancelado",
        "dt_carga",
        "_source_system",
    ],
    hash_column_name="_record_hash",
)

# ============================================================
# silver.beneficiario_rejeitado
# ============================================================

df_rejeitados = (
    df_rejeitados
    .select(
        "id_cmpt_movel",
        "cd_operadora",
        "cd_municipio",
        "cd_plano",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
        "_source_path",
        "_bronze_record_hash",
        "_bronze_ingested_at",
        "_source_system",
        "_batch_id",
        "_silver_ingested_at",
        "_layer",
        "_rejection_reason",
        "_rejected_at",
    )
)

df_rejeitados = add_record_hash(
    df=df_rejeitados,
    payload_columns=[
        "id_cmpt_movel",
        "cd_operadora",
        "cd_municipio",
        "cd_plano",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
        "_bronze_record_hash",
        "_rejection_reason",
    ],
    hash_column_name="_record_hash",
)

# ============================================================
# Escrita nas tabelas Silver
# ============================================================

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_operadora,
    target_table=SILVER_OPERADORA_TABLE,
    key_columns=["cd_operadora"],
    source_view="vw_silver_operadora_incremental",
    compare_hash_column="_record_hash",
    non_update_columns=[],
)

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_municipio,
    target_table=SILVER_MUNICIPIO_TABLE,
    key_columns=["cd_municipio"],
    source_view="vw_silver_municipio_incremental",
    compare_hash_column="_record_hash",
    non_update_columns=[],
)

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_plano,
    target_table=SILVER_PLANO_TABLE,
    key_columns=["cd_operadora", "cd_plano"],
    source_view="vw_silver_plano_incremental",
    compare_hash_column="_record_hash",
    non_update_columns=[],
)

if FULL_SNAPSHOT_REPLACE:
    replace_iceberg_by_partition_values(
        spark=spark,
        source_df=df_movimento,
        target_table=SILVER_MOVIMENTO_TABLE,
        partition_columns=["id_cmpt_movel"],
        source_view="vw_silver_movimento_snapshot",
        keys_view="vw_silver_movimento_snapshot_keys",
    )
else:
    merge_iceberg_by_keys(
        spark=spark,
        source_df=df_movimento,
        target_table=SILVER_MOVIMENTO_TABLE,
        key_columns=MOVIMENTO_BUSINESS_KEY,
        source_view="vw_silver_movimento_incremental",
        compare_hash_column="_record_hash",
        non_update_columns=[],
    )

replace_iceberg_by_partition_values(
    spark=spark,
    source_df=df_rejeitados,
    target_table=SILVER_REJEITADOS_TABLE,
    partition_columns=["_batch_id"],
    source_view="vw_silver_rejeitados_batch",
    keys_view="vw_silver_rejeitados_batch_keys",
)

# ============================================================
# Validação final
# ============================================================

spark.sql(f"SHOW TABLES IN {SILVER_DB}").show(truncate=False)

spark.sql(f"""
SELECT *
FROM {SILVER_MOVIMENTO_TABLE}
LIMIT 10
""").show(truncate=False)
